# 情報数学Ⅲ 第14回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- 「編集者」ではなく「**閲覧者**」に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

In [ ]:
# 環境セットアップ（Colab用）
!pip install japanize_matplotlib

In [ ]:
# 使用するライブラリのインポート
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gamma
import seaborn as sns

import japanize_matplotlib

## 例題1：年齢と年収をもう一度・・・



In [ ]:
# 年齢と収入のデータを含むリスト
data = [
    {"name": "A", "age": 38, "income": 500},
    {"name": "B", "age": 30, "income": 270},
    {"name": "C", "age": 28, "income": 450},
    {"name": "D", "age": 55, "income": 400},
    {"name": "E", "age": 38, "income": 250},
    {"name": "F", "age": 48, "income": 600},
    {"name": "G", "age": 30, "income": 400},
    {"name": "H", "age": 25, "income": 300},
    {"name": "I", "age": 56, "income": 800},
    {"name": "J", "age": 42, "income": 530}
]

### データフレームにする


In [ ]:
df = pd.DataFrame(data)
df.head()

### 回帰分析の実施

In [ ]:
# 線形回帰
linear_model = smf.ols("income ~ age", data=df).fit()

# 予測値をデータフレームに追加
df["pred_linear"] = linear_model.predict(df)

# 線形回帰の結果の概要
print(linear_model.summary())

In [ ]:
# プロット
plt.figure(figsize=(10, 6))
plt.scatter(df["age"], df["income"], label="実データ", color="blue")
plt.plot(df["age"], df["pred_linear"], label="線形回帰", color="red")
plt.xlabel("年齢")
plt.ylabel("年収")
plt.title("年齢と年収の関係")
plt.legend()
plt.grid()
plt.show()

### ガンマ回帰で使う関数：対数関数

In [ ]:
#　対数関数のグラフ（説明用）

# x > 0 の範囲で値を生成
x = np.linspace(0.1, 10, 400)
y = np.log(x)

# グラフの描画
plt.figure(figsize=(8, 5))
plt.plot(x, y, label=r"$\log(x)$", color="blue")
plt.axhline(0, color="gray", linestyle="--", linewidth=1)  # y=0 の線
plt.axvline(1, color="gray", linestyle="--", linewidth=1)  # x=1 の線
plt.title("対数関数 $\\log(x)$ のグラフ", fontsize=14)
plt.xlabel("x", fontsize=12)
plt.ylabel("log(x)", fontsize=12)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### ガンマ回帰で使う確率分布：ガンマ分布

In [ ]:
# x軸の値（正の範囲）
x = np.linspace(0.01, 20, 500)

# ガンマ分布のパラメータ例（shape=k, scale=θ）
params = [
    {"k": 1, "theta": 2},
    {"k": 2, "theta": 2},
    {"k": 3, "theta": 1},
    {"k": 9, "theta": 0.5},
]

# グラフの描画
plt.figure(figsize=(10, 6))
for param in params:
    k = param["k"]
    theta = param["theta"]
    y = gamma.pdf(x, a=k, scale=theta)
    label = fr"$k={k}, \theta={theta}$"
    plt.plot(x, y, label=label)

plt.title("ガンマ分布の確率密度関数", fontsize=14)
plt.xlabel("x", fontsize=12)
plt.ylabel("確率密度", fontsize=12)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### ガンマ回帰の実施

In [ ]:
# ガンマ回帰（logリンク）
# 目的変数はガンマ分布に従うと仮定

# リンク関数に対数関数を使用
gamma_model = smf.glm("income ~ age", data=df,
                      family=sm.families.Gamma(link=sm.families.links.Log())).fit()

# 参考：リンク関数の指定を省略すると，デフォルトの1/xとなる．少し結果は変わる）
#gamma_model = smf.glm("income ~ age", data=df,
#                      family=sm.families.Gamma()).fit()

# 予測値をデータフレームに追加
df["pred_gamma"] = gamma_model.predict(df)

# ガンマ回帰の結果の概要
print(gamma_model.summary())


### 線形回帰とガンマ回帰の結果を図示

In [ ]:
# データを年齢順に並べる
df_sorted = df.sort_values("age")

plt.figure(figsize=(10, 6))
plt.scatter(df["age"], df["income"], color="black", label="実データ")
plt.plot(df_sorted["age"], df_sorted["pred_linear"], "r--", label="線形回帰")
plt.plot(df_sorted["age"], df_sorted["pred_gamma"], "b-", label="ガンマ回帰")
plt.xlabel("年齢")
plt.ylabel("収入")
plt.title("年齢と年収の関係: 線形回帰とガンマ回帰")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 例題：ポアソン回帰

教科書の例題を使用

In [ ]:
# 必要なデータファイルを取得
import requests

base_url = "https://raw.githubusercontent.com/logics-of-blue/book-python-stats-2nd/refs/heads/main/book-data/"
filenames = [
    "9-4-1-poisson-regression.csv"
]

for filename in filenames:
    url = base_url + filename
    print(f"Downloading {filename}...")
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
    else:
        print(f"Failed to download {filename}: {response.status_code}")

In [ ]:
# データの読み込み
beer = pd.read_csv('9-4-1-poisson-regression.csv')
print(beer.head(3))

In [ ]:
# モデル化
mod_pois = smf.glm('beer_number ~ temperature', beer, 
                   family=sm.families.Poisson()).fit()
print(mod_pois.summary())

In [ ]:
# 予測値の作成
x_plot = np.arange(0, 37)
pred = mod_pois.predict(pd.DataFrame({'temperature': x_plot}))

# 散布図
sns.scatterplot(x='temperature', y='beer_number',
                data=beer, color='black')
# 回帰曲線を上書き
sns.lineplot(x=x_plot, y=pred, color='black')

### 回帰係数の解釈

In [ ]:
# 気温が1℃のときの販売個数の期待値
exp_val_1 = pd.DataFrame({'temperature': [1]})
pred_1 = mod_pois.predict(exp_val_1)

# 気温が2℃のときの販売個数の期待値
exp_val_2 = pd.DataFrame({'temperature': [2]})
pred_2 = mod_pois.predict(exp_val_2)

# 気温が1℃上がると、販売個数は何倍になるか
round(pred_2 / pred_1, 3)

In [ ]:
# 係数のexpをとる
round(np.exp(mod_pois.params['temperature']), 3)

## 演習：ポアソン回帰

バイクシェアリングの例

* 目的変数：cnt（その日のレンタル数）
* 説明変数：temp（気温→注意：正規化済み）、season、weekday、holiday など

In [ ]:
url = "https://raw.githubusercontent.com/ggszk/ggszk-lab-public/refs/heads/main/csv/day.csv"
df = pd.read_csv(url)
df.head()

### 気温とレンタル数の関係をポアソン回帰で調べよう

* 注意：この例は，あまりうまくモデル化はできていない．

### データを可視化しよう


### 結果を解釈して結論を書こう

このセルを編集して，以下を自分の言葉で書こう．

- 回帰係数の解釈（係数のexpをとると何がわかるか．tempは正規化済みであることに注意）
- 気温とレンタル数の関係についての結論（1〜2文）